### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from matplotlib import pyplot as plt 

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_csv  = "../Data/csvExtract/"
path_timeseries = "../Data/Output"

### Demographics

In [3]:
cohort_df = pd.read_csv(path_csv + "demographic.csv")
cohort_df[['stay_id']] = cohort_df[['stay_id']].astype(int)
cohort_df.head(2)

### Comorbidities

In [4]:
comorbidities = pd.read_csv(path_csv + "comorbidities.csv")
comorbidities = pd.merge(cohort_df, comorbidities, on=['hadm_id'], how='left')
comorbidities.drop(columns=['gender', 'age', 'race', 'admission_type', 'admission_location', 'insurance', 
                            'first_careunit', 'admittime', 'dischtime', 'intime', 'outtime', 'icuLos_h', 
                            'icuLos_d', 'hospLos_h', 'hospLos_d', 'deathtime', 'icu_expire_flag',
                            'hospital_expire_flag', 'death_time_disch'], inplace=True)
comorbidities = comorbidities.fillna(0)

In [5]:
comorbidities.head(2)

### Feature selection

In [6]:
total_icustay = cohort_df.stay_id.nunique()

In [7]:
def feature_selection(df, all_icustay, threshold):
    
    df1 = df[['stay_id', 'label']]
    df1 = df1.drop_duplicates()
    feature_list = []
    percent_list = []
    
    for feature in df1.label.unique():
        percentage = round((df[df.label == feature].stay_id.nunique())/all_icustay, 2)
        if percentage >= threshold:
            feature_list.append(feature)
            percent_list.append(percentage)
        
    columns = {'variable':feature_list,'percent':percent_list}
    frame = pd.DataFrame(columns)
    frame = frame.sort_values(by='percent', ascending=False).reset_index(drop=True)
    list_features = list(frame.variable.unique())
    
    return frame, list_features

### Lab event

In [8]:
lab_df = pd.read_csv(path_csv  + "lab_event.csv")
lab_df.head(2)

In [9]:
lab_var = [

'Temperature',
    
'PT',
'PTT',
'INR(PT)',

'pH',
'Lactate',
'Lactate Dehydrogenase (LD)',
'Base Excess',
'Anion Gap',
'Bicarbonate',
'Creatinine',
'Hematocrit', 'Hematocrit, Calculated',
'Hemoglobin',
'Bilirubin, Total',
'Bilirubin, Direct',
'Bilirubin, Indirect',
'Urea Nitrogen',

'MCH',
'MCHC',
'MCV',

'RDW',
'RBC',
'WBC',
    
'Red Blood Cells',
'White Blood Cells',
    
'Platelet Count',
'Glucose', 'Glucose, Pleural', 'Glucose, Ascites', 'Glucose, Body Fluid', 

'Ammonia',
'Magnesium',
'Phosphate',
'Alkaline Phosphatase',
'Potassium',
'Sodium', 'Sodium, Whole Blood', 
'Chloride', 'Chloride, Whole Blood',
'Free Calcium', 
'Calcium, Total', 
 
'Cholesterol, Total',
'C-Reactive Protein',

'pO2',
'pCO2',

'Alanine Aminotransferase (ALT)',
'Asparate Aminotransferase (AST)',
    
'Bands',
'Polys',
'Amylase', 'Amylase, Ascites', 'Amylase, Pleural', 'Amylase, Body Fluid',
'Lipase', 'Lipase, Ascites', 'Lipase, Pleural', 'Lipase, Body Fluid',

'Lymphocytes', 
'Monocytes',
'Eosinophils', 
'Basophils', 
'Neutrophils', 

'Absolute Lymphocyte Count',
'Absolute Monocyte Count',
'Absolute Eosinophil Count',
'Absolute Basophil Count', 
'Absolute Neutrophil Count', 

'O2 Flow',
'Oxygen',
'Oxygen Saturation', 
'Calculated Total CO2',

'Albumin',
'Troponin T', 
'Troponin I',
'Vancomycin',
'Triglycerides', 
'Fibrinogen, Functional', 
'Transferrin',
'Ferritin',    
'Cortisol',
'Protein',
'Protein, Total', 'Total Protein, Pleural', 'Total Protein, Ascites', 'Total Protein, Body Fluid',     

'PEEP', 
'Tidal Volume',
'Intubated',
'Ventilator', 
'Ventilation Rate',
    
'Granulocyte Count',
'Promyelocytes',
'Metamyelocytes',
'Myelocytes',
'Ovalocytes',    

'Creatine Kinase (CK)',
'Creatine Kinase, MB Isoenzyme', 
'CK-MB Index', 

]

In [10]:
lab_df = lab_df[lab_df.label.isin(lab_var)]
lab_df = lab_df[lab_df.value.notnull()]

In [11]:
def non_float_items(lst):
    result = []
    for item in lst:
        try:
            float(item)
        except ValueError:
            result.append(item)
    return result

In [12]:
for var in lab_var:
    if var not in ['Ventilator']:
        try:
            sub_df = lab_df[lab_df.label == var]
            values = sub_df.value.astype(float)
            
        except ValueError:
            sub_df = lab_df[lab_df.label == var]
            uniq_val = sub_df.value.unique()
            non_float_val = non_float_items(uniq_val)
            lab_df.loc[(lab_df['label'] == var)  & (lab_df['value'].isin(non_float_val)), 'value'] = np.nan
            print(var)
            print("Faild")
            print("************")
            
lab_df = lab_df[lab_df.value.notnull()]

PT
Faild
************
PTT
Faild
************
INR(PT)
Faild
************
Glucose
Faild
************
Fibrinogen, Functional
Faild
************
Protein
Faild
************


In [13]:
lab_df.loc[lab_df['label'] == 'Hematocrit, Calculated' , 'label'] = 'Hematocrit'

lab_df.loc[lab_df['label'] == 'Glucose, Pleural' ,    'label'] = 'Glucose'
lab_df.loc[lab_df['label'] == 'Glucose, Ascites' ,    'label'] = 'Glucose'
lab_df.loc[lab_df['label'] == 'Glucose, Body Fluid' , 'label'] = 'Glucose'

lab_df.loc[lab_df['label'] == 'Sodium, Whole Blood'    , 'label'] = 'Sodium'
lab_df.loc[lab_df['label'] == 'Chloride, Whole Blood'  , 'label'] = 'Chloride'

lab_df.loc[lab_df['label'] == 'Alanine Aminotransferase (ALT)'     , 'label'] = 'ALT'
lab_df.loc[lab_df['label'] == 'Asparate Aminotransferase (AST)'    , 'label'] = 'AST'

lab_df.loc[lab_df['label'] == 'Amylase, Ascites' ,    'label'] = 'Amylase'
lab_df.loc[lab_df['label'] == 'Amylase, Pleural' ,    'label'] = 'Amylase'
lab_df.loc[lab_df['label'] == 'Amylase, Body Fluid' , 'label'] = 'Amylase'

lab_df.loc[lab_df['label'] == 'Lipase, Ascites' ,    'label'] = 'Lipase'
lab_df.loc[lab_df['label'] == 'Lipase, Pleural' ,    'label'] = 'Lipase'
lab_df.loc[lab_df['label'] == 'Lipase, Body Fluid' , 'label'] = 'Lipase'

lab_df.loc[lab_df['label'] == 'Protein, Total' ,            'label'] = 'Total Protein'
lab_df.loc[lab_df['label'] == 'Total Protein, Pleural' ,    'label'] = 'Total Protein'
lab_df.loc[lab_df['label'] == 'Total Protein, Ascites' ,    'label'] = 'Total Protein'
lab_df.loc[lab_df['label'] == 'Total Protein, Body Fluid' , 'label'] = 'Total Protein'

lab_df.loc[lab_df['label'] == 'Urea Nitrogen' , 'label'] = 'BUN'
lab_df.loc[lab_df['label'] == 'Alkaline Phosphatase' , 'label'] = 'Alkaline Phosphate'
lab_df.loc[lab_df['label'] == 'Lactate Dehydrogenase (LD)' , 'label'] = 'Lactate Dehydrogenase(LDH)'
lab_df.loc[lab_df['label'] == 'Free Calcium'     , 'label'] = 'Ionized Calcium'
lab_df.loc[lab_df['label'] == 'C-Reactive Protein'     ,  'label'] = 'C-Reactive Protein (CRP)'
lab_df.loc[lab_df['label'] == 'Calculated Total CO2'    , 'label'] = 'Total CO2'
lab_df.loc[lab_df['label'] == 'Triglycerides' , 'label'] = 'Triglyceride'
lab_df.loc[lab_df['label'] == 'Fibrinogen, Functional' , 'label'] = 'Fibrinogen'
lab_df.loc[lab_df['label'] == 'Vancomycin'    , 'label'] = 'Vancomycin (Trough)'

lab_df.loc[lab_df['label'] == 'Lymphocytes'    , 'label'] = 'Differential-Lymphs'
lab_df.loc[lab_df['label'] == 'Monocytes'    ,   'label'] = 'Differential-Monos'
lab_df.loc[lab_df['label'] == 'Eosinophils'    , 'label'] = 'Differential-Eos'
lab_df.loc[lab_df['label'] == 'Basophils'    ,   'label'] = 'Differential-Basos'
lab_df.loc[lab_df['label'] == 'Neutrophils'    , 'label'] = 'Differential-Neuts'
lab_df.loc[lab_df['label'] == 'Bands'    ,       'label'] = 'Differential-Bands'
lab_df.loc[lab_df['label'] == 'Polys'    ,       'label'] = 'Differential-Polys'

lab_df.loc[lab_df['label'] == 'Creatine Kinase, MB Isoenzyme' , 'label'] = 'Creatine Kinase, MB (CK-MB)'

In [14]:
lab_df.head(2)

### Chart event

In [15]:
chart_df = pd.read_csv(path_csv  + "chart_event.csv")
chart_df.head(2)

In [16]:
chart_var = [
    
'Heart Rhythm',
'Heart Rate',
    
'Temperature Fahrenheit',
'Temperature Celsius',
'Skin Temperature',

'Respiratory Rate', 
'Respiratory Rate (spontaneous)',
'Respiratory Rate (Total)', 
'Respiratory Rate (Set)',

'Non Invasive Blood Pressure mean',
'Non Invasive Blood Pressure diastolic', 
'Non Invasive Blood Pressure systolic',
    
'Arterial Blood Pressure mean', 'ART BP Mean',
'Arterial Blood Pressure diastolic', 'ART BP Diastolic',
'Arterial Blood Pressure systolic', 'ART BP Systolic',

'Pulmonary Artery Pressure mean', 'PA mean pressure (PA Line)',
'Pulmonary Artery Pressure diastolic', 'PA diastolic pressure(PA Line)',
'Pulmonary Artery Pressure systolic', 'PA systolic pressure(PA Line)',

'Arterial O2 pressure', 'Venous O2 Pressure',
'Arterial CO2 Pressure', 'Venous CO2 Pressure',
'Mean Airway Pressure',
    
'Pain Level', 'Pain (0-10)', 'Pain Level Response', 'Pain Level Response-b',
'Pain Present',

'GCS - Eye Opening',
'GCS - Verbal Response',
'GCS - Motor Response', 

'Mental status',
'Richmond-RAS Scale',
'Goal Richmond-RAS Scale',
'Risk for Falls',
    
'Delirium',
'Delirium assessment',
'CAM-ICU MS Change', 'CAM-ICU MS change',
'CAM-ICU RASS LOC',
'CAM-ICU Inattention', 
'CAM-ICU Altered LOC', 
'CAM-ICU Disorganized thinking',
    
'Sedation goal',

'Flow Rate (L/min)',
'O2 Flow', 'O2 Flow (additional cannula)',
'Inspired O2 Fraction', 'FiO2 (CH)', 
'O2 saturation pulseoxymetry', 
'Arterial O2 Saturation', 'SaO2 < 90% > 2 min',   
'SvO2', 
    
'Admission Weight (Kg)',
'Admission Weight (lbs.)',

'Height', 'Height (cm)',    
    
'Glucose (serum)',
'Glucose finger stick (range 70-100)',
'Glucose (whole blood)', 'Glucose (whole blood) (soft)',

'Central Venous Pressure',    
'EtCO2', 
'EtCO2 Clinical indication', 
    
'ALT',
'AST',
    
'INR',
'PTT',
'WBC',
'Platelet Count', 

'Magnesium',
'Phosphorous',
'Sodium (serum)', 'Sodium (serum) (soft)', 'Sodium (whole blood)', 'Sodium (whole blood) (soft)',
'Chloride (serum)', 'Chloride (serum) (soft)', 'Chloride (whole blood)', 'Chloride (whole blood) (soft)',
'Potassium (serum)', 'Potassium (whole blood)', 
'Ionized Calcium',
'Calcium non-ionized',  
'Alkaline Phosphate', 

'BUN',
'Anion gap',
'Lactic Acid',
'HCO3 (serum)',
'Creatinine (serum)',
'Hemoglobin',
'Hematocrit (serum)',
'Direct Bilirubin',
'Total Bilirubin',
'Arterial Base Excess',
'PH (Arterial)', 'PH (Venous)', 'PH (SOFT)',

'LDH',
'Fibrinogen',  
'Troponin-T',
'Plateau Pressure', 
'Triglyceride', 
'Vancomycin (Trough)',   
'Vancomycin (Peak)', 
'Vancomycin (Random)', 

'PEEP set',
'Total PEEP Level',
    
'Tidal Volume (set)',
'Tidal Volume (observed)',
'Tidal Volume (spontaneous)',

'Ventilator Mode',
'Ventilator Type',
    
'Differential-Eos',
'Differential-Basos',  
'Differential-Lymphs', 
'Differential-Neuts', 
'Differential-Monos',
'Differential-Bands',
    
'Absolute Count - Eos', 
'Absolute Count - Basos',
'Absolute Count - Lymphs', 
'Absolute Count - Neuts',
'Absolute Count - Monos', 
    
'Total Protein', 
'C Reactive Protein (CRP)',    
'Ammonia', 'Ammonia (SOFT)',
'Cortisol', 
'Amylase', 
'Lipase',
'Albumin', 
'CO2 production',
'TCO2 (calc) Venous', 'TCO2 (calc) Arterial',
    
'Cardiac Output (thermodilution)', 'Cardiac Output (CCO)', 'Cardiac Output (CO NICOM)',

'CK (CPK)',
'CK-MB',
'CK-MB fraction (%)',
    
'O2 Delivery Device(s)',

]

In [17]:
chart_df = chart_df[chart_df.label.isin(chart_var)]
chart_df = chart_df[chart_df.value.notnull()]

In [18]:
def non_float_items(lst):
    result = []
    for item in lst:
        try:
            float(item)
        except ValueError:
            result.append(item)
    return result

In [19]:
for var in chart_var:
    if var not in ['Heart Rhythm', 'Ventilator Mode', 'Ventilator Type',
                   'O2 Delivery Device(s)', 'EtCO2 Clinical indication', ]:
        try:
            sub_df = chart_df[chart_df.label == var]
            values = sub_df.value.astype(float)
            
        except ValueError:
            sub_df = chart_df[chart_df.label == var]
            uniq_val = sub_df.value.unique()
            non_float_val = non_float_items(uniq_val)
            chart_df.loc[(chart_df['label'] == var)  & (chart_df['value'].isin(non_float_val)), 'value'] = np.nan
            print(var)
            print("Faild")
            print("************")
            
chart_df = chart_df[chart_df.value.notnull()]

Pain Level
Faild
************
Pain Level Response
Faild
************


In [20]:
chart_df.loc[chart_df['label'] == 'Temperature Fahrenheit' , 'value'] = (chart_df[chart_df['label'] == 'Temperature Fahrenheit'].value.astype(float) - 32) * 5 / 9

chart_df.loc[chart_df['label'] == 'Admission Weight (lbs.)' , 'value'] = (chart_df[chart_df['label'] == 'Admission Weight (lbs.)'].value.astype(float) * 0.45)

chart_df.loc[chart_df['label'] == 'Height' , 'value'] = (chart_df[chart_df['label'] == 'Height'].value.astype(float) * 2.54)

In [21]:
chart_df.loc[chart_df['label'] == 'Temperature Fahrenheit' , 'label'] = 'Temperature'
chart_df.loc[chart_df['label'] == 'Temperature Celsius' ,    'label'] = 'Temperature'

chart_df.loc[chart_df['label'] == 'Respiratory Rate (spontaneous)'  , 'label'] = 'Respiratory Rate'

chart_df.loc[chart_df['label'] == 'ART BP Mean',      'label'] = 'Arterial Blood Pressure mean'
chart_df.loc[chart_df['label'] == 'ART BP Diastolic', 'label'] = 'Arterial Blood Pressure diastolic'
chart_df.loc[chart_df['label'] == 'ART BP Systolic',  'label'] = 'Arterial Blood Pressure systolic'

chart_df.loc[chart_df['label'] == 'PA mean pressure (PA Line)',     'label'] = 'Pulmonary Artery Pressure mean'
chart_df.loc[chart_df['label'] == 'PA diastolic pressure(PA Line)', 'label'] = 'Pulmonary Artery Pressure diastolic'
chart_df.loc[chart_df['label'] == 'PA systolic pressure(PA Line)',  'label'] = 'Pulmonary Artery Pressure systolic'

chart_df.loc[chart_df['label'] == 'Venous O2 Pressure'  ,    'label'] = 'pO2'
chart_df.loc[chart_df['label'] == 'Arterial O2 pressure'   , 'label'] = 'pO2'
chart_df.loc[chart_df['label'] == 'Arterial CO2 Pressure'  , 'label'] = 'pCO2'
chart_df.loc[chart_df['label'] == 'Venous CO2 Pressure'   ,  'label'] = 'pCO2'

chart_df.loc[chart_df['label'] == 'Pain (0-10)'   ,           'label'] = 'Pain Level'
chart_df.loc[chart_df['label'] == 'Pain Level Response'   ,   'label'] = 'Pain Level'
chart_df.loc[chart_df['label'] == 'Pain Level Response-b'   , 'label'] = 'Pain Level'

chart_df.loc[chart_df['label'] == 'CAM-ICU MS change'   , 'label'] = 'CAM-ICU MS Change'

chart_df.loc[chart_df['label'] == 'O2 Flow (additional cannula)'   , 'label'] = 'O2 Flow'

chart_df.loc[chart_df['label'] == 'Inspired O2 Fraction'   , 'label'] = 'FiO2'
chart_df.loc[chart_df['label'] == 'FiO2 (CH)'   ,            'label'] = 'FiO2'

chart_df.loc[chart_df['label'] == 'O2 saturation pulseoxymetry'   , 'label'] = 'SpO2'
chart_df.loc[chart_df['label'] == 'Arterial O2 Saturation'   , 'label'] = 'Oxygen Saturation'
chart_df.loc[chart_df['label'] == 'SaO2 < 90% > 2 min'   , 'label'] = 'Oxygen Saturation'

chart_df.loc[chart_df['label'] == 'Admission Weight (lbs.)'   , 'label'] = 'Admission Weight (Kg)'
chart_df.loc[chart_df['label'] == 'Height'   , 'label'] = 'Height (cm)'

chart_df.loc[chart_df['label'] == 'Glucose (serum)'                      , 'label'] = 'Glucose'
chart_df.loc[chart_df['label'] == 'Glucose (whole blood)'                , 'label'] = 'Glucose'
chart_df.loc[chart_df['label'] == 'Glucose finger stick (range 70-100)'  , 'label'] = 'Glucose'
chart_df.loc[chart_df['label'] == 'Glucose (whole blood) (soft)'         , 'label'] = 'Glucose'

chart_df.loc[chart_df['label'] == 'Sodium (serum)'       ,        'label'] = 'Sodium'
chart_df.loc[chart_df['label'] == 'Sodium (serum) (soft)' ,       'label'] = 'Sodium'
chart_df.loc[chart_df['label'] == 'Sodium (whole blood) (soft)' , 'label'] = 'Sodium'
chart_df.loc[chart_df['label'] == 'Sodium (whole blood)' ,        'label'] = 'Sodium'

chart_df.loc[chart_df['label'] == 'Chloride (serum)'    ,          'label'] = 'Chloride'
chart_df.loc[chart_df['label'] == 'Chloride (serum) (soft)'    ,   'label'] = 'Chloride'
chart_df.loc[chart_df['label'] == 'Chloride (whole blood)'    ,    'label'] = 'Chloride'
chart_df.loc[chart_df['label'] == 'Chloride (whole blood) (soft)', 'label'] = 'Chloride'

chart_df.loc[chart_df['label'] == 'Potassium (serum)'       , 'label'] = 'Potassium'
chart_df.loc[chart_df['label'] == 'Potassium (whole blood)' , 'label'] = 'Potassium'

chart_df.loc[chart_df['label'] == 'PH (Arterial)' , 'label'] = 'pH'
chart_df.loc[chart_df['label'] == 'PH (Venous)' ,   'label'] = 'pH'
chart_df.loc[chart_df['label'] == 'PH (SOFT)' ,     'label'] = 'pH'

chart_df.loc[chart_df['label'] == 'PEEP set'     , 'label'] = 'PEEP (Set)'
chart_df.loc[chart_df['label'] == 'Total PEEP Level'  , 'label'] = 'PEEP'

chart_df.loc[chart_df['label'] == 'Tidal Volume (set)'     , 'label'] = 'Tidal Volume (Set)'
chart_df.loc[chart_df['label'] == 'Tidal Volume (observed)'     , 'label'] = 'Tidal Volume'
chart_df.loc[chart_df['label'] == 'Tidal Volume (spontaneous)'  , 'label'] = 'Tidal Volume'

chart_df.loc[chart_df['label'] == 'TCO2 (calc) Venous'   ,   'label'] = 'Total CO2'
chart_df.loc[chart_df['label'] == 'TCO2 (calc) Arterial'   , 'label'] = 'Total CO2'

chart_df.loc[chart_df['label'] == 'Ammonia (SOFT)'   , 'label'] = 'Ammonia'
chart_df.loc[chart_df['label'] == 'INR'   , 'label'] = 'INR(PT)'

chart_df.loc[chart_df['label'] == 'Lactic Acid'   , 'label'] = 'Lactate'
chart_df.loc[chart_df['label'] == 'LDH' , 'label'] = 'Lactate Dehydrogenase(LDH)'

chart_df.loc[chart_df['label'] == 'Arterial Base Excess' , 'label'] = 'Base Excess'
chart_df.loc[chart_df['label'] == 'Anion gap'     , 'label'] = 'Anion Gap'
chart_df.loc[chart_df['label'] == 'HCO3 (serum)'        , 'label'] = 'Bicarbonate'
chart_df.loc[chart_df['label'] == 'Creatinine (serum)'  , 'label'] = 'Creatinine'
chart_df.loc[chart_df['label'] == 'Hematocrit (serum)'  , 'label'] = 'Hematocrit'

chart_df.loc[chart_df['label'] == 'Direct Bilirubin'   , 'label'] = 'Bilirubin, Direct'
chart_df.loc[chart_df['label'] == 'Total Bilirubin'    , 'label'] = 'Bilirubin, Total'

chart_df.loc[chart_df['label'] == 'Phosphorous'   , 'label'] = 'Phosphate'
chart_df.loc[chart_df['label'] == 'C Reactive Protein (CRP)'   , 'label'] = 'C-Reactive Protein (CRP)'
chart_df.loc[chart_df['label'] == 'Troponin-T', 'label'] = 'Troponin T'

chart_df.loc[chart_df['label'] == 'Absolute Count - Eos'    , 'label'] = 'Absolute Eosinophil Count'
chart_df.loc[chart_df['label'] == 'Absolute Count - Basos'  , 'label'] = 'Absolute Basophil Count'
chart_df.loc[chart_df['label'] == 'Absolute Count - Lymphs' , 'label'] = 'Absolute Lymphocyte Count'
chart_df.loc[chart_df['label'] == 'Absolute Count - Neuts'  , 'label'] = 'Absolute Neutrophil Count'
chart_df.loc[chart_df['label'] == 'Absolute Count - Monos'  , 'label'] = 'Absolute Monocyte Count'

chart_df.loc[chart_df['label'] == 'Cardiac Output (thermodilution)' , 'label'] = 'Cardiac Output (CO)'
chart_df.loc[chart_df['label'] == 'Cardiac Output (CCO)'            , 'label'] = 'Cardiac Output (CO)'
chart_df.loc[chart_df['label'] == 'Cardiac Output (CO NICOM)'       , 'label'] = 'Cardiac Output (CO)'

chart_df.loc[chart_df['label'] == 'CK (CPK)'             , 'label'] = 'Creatine Kinase (CK)'
chart_df.loc[chart_df['label'] == 'CK-MB fraction (%)'   , 'label'] = 'CK-MB Index'
chart_df.loc[chart_df['label'] == 'CK-MB'                , 'label'] = 'Creatine Kinase, MB (CK-MB)'

In [22]:
chart_df.head(2)

### Prescriptions

In [23]:
prescriptions = pd.read_csv(path_csv + "prescriptions.csv")
prescriptions.head(2)

### Date&Time events

In [24]:
datetimeevents = pd.read_csv(path_csv  + "datetimeevents.csv")
datetimeevents.head(2)

### Procedure events

In [25]:
procedureevents = pd.read_csv(path_csv  + "procedureevents.csv")
procedureevents.head(2)

### Output events

In [26]:
outputevents = pd.read_csv(path_csv  + "outputevents.csv")
outputevents.head(2)

### Input events

In [27]:
inputevents = pd.read_csv(path_csv + "inputevents.csv")
inputevents.head(2)

### Blood culture

In [28]:
culture_df = pd.read_csv(path_csv + "culture_event.csv")
culture_df.head(2)

,subject_id,hadm_id,stay_id,charttime,label,value


### Microbiology test 

In [29]:
microbio_df = pd.read_csv(path_csv + "microbio_event.csv")
microbio_df.head(2)

### All Tables

In [30]:
tables = [lab_df, chart_df, prescriptions, datetimeevents, procedureevents, outputevents, inputevents, culture_df,
          microbio_df]

In [31]:
all_tables = pd.concat(tables)
all_tables[['stay_id']] = all_tables[['stay_id']].astype(int)
all_tables = all_tables.sort_values(by=['stay_id', 'charttime'], axis=0)
all_tables.reset_index(inplace=True, drop=True)

In [32]:
all_tables.head(2)

### Save Data

In [33]:
def cohort_stay_id(frame):
    cohort = frame.stay_id.unique()
    return cohort

### ICU Admission Information

In [34]:
def break_up_admission_by_unit_stay(admission, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        admission.loc[admission.stay_id == icu_stay_id].to_csv(os.path.join(dn, 'admission.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [ ]:
icu_stayid_adm  = cohort_stay_id(cohort_df)
break_up_admission_by_unit_stay(cohort_df, path_timeseries, icu_stayid_adm, verbose=1)

### ICU patients comorbidities

In [36]:
def break_up_comorbidities_by_unit_stay(comorbidity, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        comorbidity.loc[comorbidity.stay_id == icu_stay_id].to_csv(os.path.join(dn, 'comorbidity.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [37]:
icu_stayid_comorbidity  = cohort_stay_id(comorbidities)
break_up_comorbidities_by_unit_stay(comorbidities, path_timeseries, icu_stayid_comorbidity, verbose=1)

StayID 10564 of 10564...DONE!


### All Tables

In [38]:
def break_up_all_tables_by_unit_stay(all_tables, output_path, stayid, verbose=1):
    
    unit_stays = stayid
    nb_unit_stays = unit_stays.shape[0]
    
    for i, icu_stay_id in enumerate(unit_stays):
        
        if verbose:
            sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
            
        dn = os.path.join(output_path, str(icu_stay_id))
        
        try:
            os.makedirs(dn)  
        except:
            pass

        all_tables.loc[all_tables.stay_id == icu_stay_id].to_csv(os.path.join(dn, 'all_tables.csv'), index=False)
    
    if verbose:
        sys.stdout.write('DONE!\n')

In [39]:
stay_id_all_tables  = cohort_stay_id(all_tables)
break_up_all_tables_by_unit_stay(all_tables, path_timeseries, stay_id_all_tables, verbose=1)

StayID 10563 of 10563...DONE!


### Save list of unique variables

In [3]:
all_variables = [
    
'Admission Weight (Kg)', 'Height (cm)',
    
'Microbio Test', 'Blood Culture', 
    
'radiology_note', 'discharge_note',  'cxr_image', 'cxr_note',  'cxr_lung', 'ecg_waveform', 'ecg_report',
    
'22 Gauge Insertion Date', '20 Gauge Insertion Date', '18 Gauge Insertion Date', 'Arterial line Insertion Date',
'Arterial line Tubing Change', 'Arterial Line Dressing Change', 'Multi Lumen Insertion Date',
'Multi Lumen Cap Change', 'Multi Lumen Dressing Change', 'Multi Lumen Tubing Change',
    
'Antibiotic_PRC', 'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC', 'Midazolam_PRC', 
'Heparin_PRC', 'Dexmedetomidine_PRC', 'Amiodarone_PRC', 'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC',
'Nicardipine_PRC', 'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 'Nitroglycerin_PRC',
'Epinephrine_PRC', 'Warfarin_PRC', 'Apixaban_PRC', 'Dabigatran_PRC', 'Rivaroxaban_PRC', 'Edoxaban_PRC',
 
'UrineOutput_IO', 'Stool_IO', 'Propofol_IO', 'Fentanyl_IO', 'Insulin_IO', 'Heparin_IO', 'Midazolam_IO',
'Dexmedetomidine_IO', 'Vassopressin_IO', 'Albumin_IO', 'Ceftriaxone_IO', 'Cefazolin_IO', 'Cefepime_IO',
'Ceftazidime_IO', 'Vancomycin_IO', 'Clindamycin_IO', 'Metronidazole_IO', 'Meropenem_IO', 'Acyclovir_IO', 
'Azithromycin_IO', 'Levofloxacin_IO', 'Micafungin_IO', 'Fluconazole_IO', 'Thiamine_IO', 'Dobutamine_IO', 
'Milrinone_IO', 'Fluids_IO', 'OralIntake_IO', 'P.O._IO', 'SodiumChloride_IO', 'IVPB_IO', 'Crystalloids_IO',
'NSIVF_IO', 'Norepinephrine_IO', 'Amiodarone_IO', 'Phenylephrine_IO', 'Epinephrine_IO', 'Nicardipine_IO',
'Pantoprazole_IO', 'Diltiazem_IO', 'Nitroglycerin_IO', 'NeuroblockAgent_IO', 'K-IV_IO', 'Ca-IV_IO', 'Ca-nonIV_IO',
'Mg-IV_IO', 'Mg-nonIV_IO', 'P-IV_IO', 'P-nonIV_IO', 'BetaBlockers_IO', 'CaBlockers_IO', 'LoopDiuretics_IO',
'TPNutrition_IO', 'PNutrition_IO', 'Dextrose_IO', 'POnutrition_IO', 'Vasopressors_IO',
    
'PT', 'PTT', 'INR(PT)', 'pH', 'Lactate', 'Lactate Dehydrogenase(LDH)', 'Base Excess',
'Anion Gap', 'Bicarbonate', 'Creatinine', 'Hematocrit',  'Hemoglobin', 'Bilirubin, Total', 'Bilirubin, Direct',
'Bilirubin, Indirect', 'BUN', 'MCH', 'MCHC', 'MCV', 'RDW', 'RBC', 'WBC', 'Red Blood Cells', 'White Blood Cells',
'Platelet Count', 'Glucose', 'Ammonia', 'Magnesium', 'Phosphate', 'Alkaline Phosphate', 'Potassium', 'Sodium', 
'Chloride', 'Calcium non-ionized', 'Ionized Calcium', 'Calcium, Total', 'Cholesterol, Total', 
'C-Reactive Protein (CRP)', 'pO2', 'pCO2', 'ALT', 'AST', 'Amylase', 'Lipase', 'Albumin', 'Troponin T', 
'Troponin I', 'Triglyceride', 'Fibrinogen', 'Transferrin', 'Ferritin', 'Cortisol', 'Protein', 'Total Protein', 
'Vancomycin (Trough)', 'Vancomycin (Peak)', 'Vancomycin (Random)', 

'Cardiac Output (CO)', 'Creatine Kinase (CK)', 'Creatine Kinase, MB (CK-MB)', 'CK-MB Index',
    
'Differential-Lymphs', 'Differential-Monos', 'Differential-Eos',  'Differential-Basos',  'Differential-Neuts',
'Differential-Bands', 'Differential-Polys', 'Absolute Lymphocyte Count', 'Absolute Monocyte Count', 
'Absolute Eosinophil Count', 'Granulocyte Count', 'Absolute Basophil Count', 'Absolute Neutrophil Count', 
'Promyelocytes', 'Metamyelocytes', 'Myelocytes', 'Ovalocytes', 
    
'Pain Level', 'Pain Present', 'GCS - Eye Opening', 'GCS - Verbal Response', 'GCS - Motor Response', 
'Mental status', 'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', 'Risk for Falls', 'Delirium',
'Delirium assessment', 'CAM-ICU MS Change', 'CAM-ICU RASS LOC', 'CAM-ICU Inattention', 'CAM-ICU Altered LOC', 
'CAM-ICU Disorganized thinking', 'Sedation goal', 'EtCO2 Clinical indication',
    
'Heart Rhythm', 'Heart Rate', 'Temperature', 'Skin Temperature', 'Respiratory Rate', 'Respiratory Rate (Set)',
'Respiratory Rate (Total)', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure diastolic', 
'Non Invasive Blood Pressure systolic', 'Arterial Blood Pressure mean', 'Arterial Blood Pressure diastolic', 
'Arterial Blood Pressure systolic', 'Pulmonary Artery Pressure mean',  'Pulmonary Artery Pressure diastolic',
'Pulmonary Artery Pressure systolic', 
        
'SpO2', 'SvO2', 'Oxygen Saturation', 'O2 Flow', 'Oxygen', 'CO2 production', 'Total CO2', 'Mean Airway Pressure',
 
'PEEP', 'PEEP (Set)', 'FiO2', 'Tidal Volume', 'Tidal Volume (Set)', 'Plateau Pressure', 
'Flow Rate (L/min)', 'Central Venous Pressure', 'EtCO2',
    
'Intubated', 'Ventilator', 'Ventilator Mode', 'Ventilator Type', 'Ventilation', 'Ventilation Rate', 
'O2 Delivery Device(s)',

]

cat_int_value = ['Intubated', 'Skin Temperature', 'Pain Level', 'Pain Present', 'GCS - Eye Opening', 
                 'GCS - Motor Response', 'GCS - Verbal Response', 'Mental status', 'Richmond-RAS Scale',
                 'Goal Richmond-RAS Scale', 'Risk for Falls', 'Delirium assessment', 'CAM-ICU MS Change',
                 'CAM-ICU RASS LOC', 'CAM-ICU Inattention', 'CAM-ICU Altered LOC', 'CAM-ICU Disorganized thinking',
                  'Microbio Test', 'Blood Culture', 'Ventilation', 'Sedation goal']

cat_text_value = ['Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator', 'O2 Delivery Device(s)'
                  'EtCO2 Clinical indication']

In [4]:
with open("../Data/csvExtract/variables.txt", "w") as f:
    for variable in all_variables:
        f.write(variable +"\n")